## 授權與來源（License & Attribution）

本課程自編／改編的程式與文字：Copyright 2026 leonjyentub；Apache License 2.0（`../LICENSE`）。
教學概念與來源參照：Aurélien Géron, `ageron/handson-mlp` Appendix A 自動微分，上游 Apache-2.0，固定 commit `47eba45aacc85feae51ba7db68dd1ca66cb25e0a`。
本檔改編／創作說明：依原書概念另編可於 CPU 執行的小型機制實驗、示範圖表與中文教學解釋；不等同於完整上游實驗，亦不表示每段程式均直接複製上游。
詳細對照：`../SOURCES.md`；授權及署名：`../THIRD_PARTY_NOTICES.md`。
其他第三方資料、圖片及模型權利按各自來源處理，不包含在本課程程式授權中。

# 24｜附錄A_自動微分與計算圖

以中央差分檢查解析梯度，並觀察步長過大或過小的誤差。

對應 `slides/24_附錄A_自動微分與計算圖.md`；從第一格依序執行即可重現投影片中的表格與圖。

**Colab 示範**：直接上傳此 `.ipynb`，從上到下執行；第一個程式格會安裝缺少的套件，所需資料在使用時自動下載。NumPy 與 scikit-learn 實驗使用 CPU；這些程式沒有可直接切換的 CUDA 後端。輸出存於目前工作目錄。


## 來源與改編界線

- Aurélien Géron，`handson-mlp/Appendix_A_autodiff.ipynb`，固定 commit `47eba45aacc85feae51ba7db68dd1ca66cb25e0a`，參考 Cells 6、17–23（數值微分與 toy computation graph）。

- `MachineLearning2025/06_ANN_from_Scratch_MNIST.py`，固定 commit `3da2cb56b9efd17d7cd34602589f392554b40603`。
- 本 Notebook 為課堂短實驗：保留核心張量、演算法或評估流程，改用 NumPy／scikit-learn 與小型資料，讓 CPU 能快速重跑。
- 圖表與數值是本檔實際執行結果，不代表原書完整模型成效。原始程式授權與歸屬見 `../NOTICE.md`。

In [ ]:
# 獨立執行：Colab 上傳此檔後，從這一格依序執行；無需掛載 Git 專案。
import importlib.util, subprocess, sys
_needed = {"numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.0,<3", "matplotlib": "matplotlib>=3.7,<4", "scipy": "scipy>=1.11,<2", "sklearn": "scikit-learn>=1.4,<2", "joblib": "joblib>=1.3,<2", "cloudpickle": "cloudpickle>=3,<4", "ipywidgets": "ipywidgets>=8,<9"}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

from pathlib import Path
import hashlib, io, json, os, tarfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# 本機在 programs/ 執行時沿用 data/、outputs/；Colab 則使用自己的工作目錄。
DATA = ROOT / ("data" if (ROOT / "pyproject.toml").exists() else "mlcourse_data")
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
SEED = 42
FAST = os.environ.get("MLCOURSE_PROFILE", "fast") != "full"
_SOURCE_COMMIT = "9e29abbbea6a3ff7250dea64c43c1d6c723095d7"
_SOURCES = {
    "housing.csv": (f"https://raw.githubusercontent.com/ageron/data/{_SOURCE_COMMIT}/housing.tgz", "2364609dc48bec7df3ba9dbb7041478e704ecddcee70ef1827ec3fc49d22c0cc"),
    "lifesat.csv": (f"https://raw.githubusercontent.com/ageron/data/{_SOURCE_COMMIT}/lifesat/lifesat.csv", "e247f7f6c19b0d8c55fb2af293f4fb3b710e31476534ad8e9b7e06ea546a2099"),
}

def setup():
    plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                         "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})
    OUT.mkdir(parents=True, exist_ok=True)
    return {"profile": "fast" if FAST else "full", "seed": SEED, "data": str(DATA)}

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

def data_path(name):
    if name not in (*_SOURCES, "mnist_784_v1.npz"):
        raise ValueError(f"未知資料檔：{name}")
    DATA.mkdir(parents=True, exist_ok=True)
    path = DATA / name
    if name in _SOURCES:
        url, expected_sha = _SOURCES[name]
        if not path.exists():
            print(f"下載 {name} …")
            with urllib.request.urlopen(url, timeout=180) as response:
                blob = response.read()
            if name == "housing.csv":
                with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as archive:
                    blob = archive.extractfile("housing/housing.csv").read()
            if hashlib.sha256(blob).hexdigest() != expected_sha:
                raise ValueError(f"資料校驗失敗：{name}")
            path.write_bytes(blob)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha:
            raise ValueError(f"本機資料校驗失敗：{name}")
        info = {"url": url, "sha256": expected_sha}
    else:
        if not path.exists():
            print("從 OpenML 下載 MNIST（首次約需數分鐘）…")
            from sklearn.datasets import fetch_openml
            ds = fetch_openml("mnist_784", version=1, as_frame=False,
                              data_home=DATA / "openml_cache", parser="auto")
            X, y = ds.data.astype(np.uint8), ds.target.astype(np.uint8)
            assert X.shape == (70000, 784) and y.shape == (70000,)
            np.savez_compressed(path, X=X, y=y)
        with np.load(path) as archive:
            assert archive["X"].shape == (70000, 784)
            assert archive["y"].shape == (70000,)
        info = {"url": "https://www.openml.org/d/554",
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    manifest_file = DATA / "manifest.json"
    manifest = json.loads(manifest_file.read_text()) if manifest_file.exists() else {}
    manifest[name] = info
    manifest_file.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
    return path

def housing_split():
    from sklearn.model_selection import train_test_split
    frame = pd.read_csv(data_path("housing.csv"))
    income = pd.cut(frame.median_income, [0, 1.5, 3, 4.5, 6, np.inf], labels=[1, 2, 3, 4, 5])
    train, test = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=income)
    assert not set(train.index) & set(test.index)
    return train, test

setup()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(SEED)

## 課堂小實驗

In [ ]:
def f(x, y): return x*x*y + y + 2
x, y, check_eps = 3.0, 4.0, 1e-5
analytic = np.array([2*x*y, x*x+1])
numeric_f = np.array([(f(x+check_eps,y)-f(x-check_eps,y))/(2*check_eps),
                      (f(x,y+check_eps)-f(x,y-check_eps))/(2*check_eps)])

# f 對 x 是二次式，中央差分會特別精準；另用非多項式 g 觀察步長誤差。
def g(v): return np.sin(v) * np.exp(v/10)
g_prime = np.exp(x/10) * (np.cos(x) + 0.1*np.sin(x))
eps_values = np.logspace(-1, -14, 14)
numeric_g = np.array([(g(x+e)-g(x-e))/(2*e) for e in eps_values])
errors = np.abs(numeric_g-g_prime)
best = int(np.argmin(errors))
print('f analytic gradient:', analytic, ' finite difference:', numeric_f)
print('g best epsilon:', eps_values[best], 'error:', errors[best])

## 執行結果圖

In [ ]:
plt.loglog(eps_values, errors, 'o-')
plt.axvline(eps_values[best], color='tab:red', ls='--', label=f'best eps={eps_values[best]:.0e}')
plt.gca().invert_xaxis(); plt.xlabel('epsilon'); plt.ylabel('gradient error')
plt.title('Finite differences fail when epsilon is too large or too small'); plt.legend()
savefig('24_autodiff_demo')
report('autodiff', {'f_analytic': analytic.tolist(), 'f_numeric': numeric_f.tolist(),
                    'g_best_epsilon': eps_values[best], 'g_best_error': errors[best]})

## 解讀與延伸

先描述圖或表直接支持的結果，再說明這個縮編實驗不能代表完整模型的哪些面向。